In [64]:
import os
from dotenv import load_dotenv
load_dotenv()

print(bool(os.getenv("OPENAI_API_KEY")))

True


In [65]:
import PyPDF2

# Open the PDF file
with open('../data/coffee_processing.pdf', 'rb') as file:
    # Create a PDF reader object
    pdf_reader = PyPDF2.PdfReader(file)
    
    # Get the number of pages
    num_pages = len(pdf_reader.pages)
    print(f"Total pages: {num_pages}")
    
    # Initialize variable to store all text
    all_text = []
    
    # Extract text from all pages
    for page_num in range(num_pages):
        page = pdf_reader.pages[page_num]
        text = page.extract_text()
        if text:  # Only add if page has text
            all_text.append(text.strip())
    
    # Combine all text with double newline between pages
    text_document = "\n\n".join(all_text)

    text_document = text_document.replace("Explore our developer-friendly HTML  to PDF API Printed using PDFCrowd HTML  to PDF", "")
    
    # Print the combined text
    print("\n=== COMBINED TEXT ===\n")
    print(text_document)
    
    # Optionally print stats
    print(f"\n\nTotal characters: {len(text_document)}")

Total pages: 4

=== COMBINED TEXT ===

Coffee Processing Methods
A Comprehensive Guide to Post-Harvest Coffee Processing
Coffee processing began in Ethiopia over 1,000 years ago.
Introduction to Coffee Processing
Coffee processing is the method used to transform freshly picked coffee cherries into green coffee
beans ready for roasting. The processing method significantly impacts the final flavor profile of the
coffee. Different regions around the world have developed unique processing techniques based on
their climate, available resources, and traditional practices. The choice of processing method can
enhance certain flavor characteristics while suppressing others, making it one of the most critical
decisions in coffee production.
Processing begins immediately after harvest, as coffee cherries are highly perishable. The outer
fruit must be removed, and the beans must be dried to prevent spoilage. The manner in which this
is accomplished varies widely, from ancient sun-drying methods to

In [66]:
import re

def clean_text(text: str) -> str:
    # Remove long dash lines
    text = re.sub(r'-{3,}', ' ', text)
    
    # Remove emojis and special symbols (keep normal punctuation)
    text = re.sub(r'[^\w\s\.\,\-\%\(\)]', ' ', text)
    
    # Remove standalone dots
    text = re.sub(r'\n\s*\.\s*\n', '\n', text)
    
    # Fix broken decimals like 10 \n .5-12 \n .5%
    text = re.sub(r'(\d)\s*\.\s*(\d)', r'\1.\2', text)
    
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text)
    
    return text.strip()


def clean_chunk(text: str) -> str:
    # Remove long dash separators
    text = re.sub(r'-{3,}', ' ', text)

    # Remove "Print to PDF"
    text = re.sub(r'Print to PDF', '', text)

    # Remove leading dot
    text = re.sub(r'^\s*\.\s*', '', text)

    # Fix broken decimal formatting
    text = re.sub(r'(\d)\s*\.\s*(\d)', r'\1.\2', text)

    # Remove duplicate fragment like "and drying, green coffee beans..."
    text = re.sub(r'and drying, green coffee beans.*?\)', '', text)

    # Collapse multiple spaces
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

cleaned_document = clean_text(text_document)

In [67]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document


text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80,
    separators=["\n\n", "\n", ". ", " "]
)
chunks = text_splitter.create_documents([cleaned_document])

print(f'Total no of chunks: {len(chunks)}', end='\n')

cleaned_documents = []

for i, chunk in enumerate(chunks):
    raw_text = chunk.page_content
    cleaned_text = clean_chunk(raw_text)
    
    doc = Document(
        page_content=cleaned_text,
        metadata={
            "source": "coffee_processing_guide",
            "chunk_id": i
        }
    )
    cleaned_documents.append(doc)

print(f"Total LangChain Documents: {len(cleaned_documents)}\n")

for doc in cleaned_documents:
    print(doc.page_content)
    print(doc.metadata)
    print("-" * 40)


Total no of chunks: 15
Total LangChain Documents: 15

Coffee Processing Methods A Comprehensive Guide to Post-Harvest Coffee Processing Coffee processing began in Ethiopia over 1,000 years ago. Introduction to Coffee Processing Coffee processing is the method used to transform freshly picked coffee cherries into green coffee beans ready for roasting. The processing method significantly impacts the final flavor profile of the coffee
{'source': 'coffee_processing_guide', 'chunk_id': 0}
----------------------------------------
Different regions around the world have developed unique processing techniques based on their climate, available resources, and traditional practices. The choice of processing method can enhance certain flavor characteristics while suppressing others, making it one of the most critical decisions in coffee production. Processing begins immediately after harvest, as coffee cherries are highly perishable. The outer fruit must be removed, and the beans must be dried to 

In [68]:
from langchain_openai import OpenAIEmbeddings
import chromadb

# Initialize embeddings model
embeddings_model = OpenAIEmbeddings(model="text-embedding-3-large")

# Set up ChromaDB client with persistent storage
chroma_client = chromadb.PersistentClient(path="../chroma_db")

In [69]:
# Delete existing collection if it exists (truncate)
try:
    chroma_client.delete_collection(name="my_documents_recursive_chunking_reranking")
    print("Existing collection deleted")
except:
    print("No existing collection found")

# Create new collection
collection = chroma_client.create_collection(
    name="my_documents_recursive_chunking_reranking",
    metadata={"description": "Document embeddings collection", "hnsw:space": "cosine"}
)

Existing collection deleted


In [71]:
# Calculate embeddings in batches and prepare data
documents = []
embeddings_list = []
ids = []

# Extract all texts first
all_texts = []
for idx, chunk in enumerate(cleaned_documents):
    text = chunk["page_content"] if isinstance(chunk, dict) else chunk.page_content
    all_texts.append(text)
    ids.append(f"doc_{idx}")

# Calculate embeddings in batches
batch_size = 100  # Adjust based on your needs and API limits
total_batches = (len(all_texts) + batch_size - 1) // batch_size

print(f"Processing {len(all_texts)} documents in {total_batches} batches...")

for i in range(0, len(all_texts), batch_size):
    batch_texts = all_texts[i:i + batch_size]
    
    # Calculate embeddings for the batch
    batch_embeddings = embeddings_model.embed_documents(batch_texts)
    
    # Add to main lists
    documents.extend(batch_texts)
    embeddings_list.extend(batch_embeddings)
    
    batch_num = (i // batch_size) + 1
    print(f"Processed batch {batch_num}/{total_batches} ({len(batch_texts)} documents)")

print(f"\nTotal embeddings calculated: {len(embeddings_list)}")

# Insert all data into ChromaDB
collection.add(
    documents=documents,
    embeddings=embeddings_list,
    ids=ids
)

print(f"Successfully added {len(documents)} documents to ChromaDB")

Processing 15 documents in 1 batches...
Processed batch 1/1 (15 documents)

Total embeddings calculated: 15
Successfully added 15 documents to ChromaDB


In [74]:
# -----------------------------
# TEST QUERY VARIATIONS
# -----------------------------

# query = "What are the environmental impacts of coffee processing?"
# query = "How does washed processing affect the environment?"
# query = "What solutions exist to reduce water usage in wet coffee processing?"
# query = "Explain wastewater management in coffee production."
query = "Sustainability practices in coffee processing methods."


# Embed query
query_embedding = embeddings_model.embed_query(query)

# Number of results
k = min(5, len(documents))

# Query Chroma
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=k,
    include=["documents", "distances"]
)

print("\nQuery:")
print(query.strip())
print("\nTop Results:")
print("=" * 100)

for rank, (doc, distance) in enumerate(
    zip(results["documents"][0], results["distances"][0]),
    start=1
):
    similarity = 1 - distance
    percentage = similarity * 100

    print(f"\nRank: {rank}")
    print(f"Distance: {distance:.4f}")
    print(f"Similarity: {similarity:.4f}")
    print(f"Confidence: {percentage:.2f}%")
    print(f'\nChunk: {doc}')
    print("=" * 100)


Query:
Sustainability practices in coffee processing methods.

Top Results:

Rank: 1
Distance: 0.3840
Similarity: 0.6160
Confidence: 61.60%

Chunk: The manner in which this is accomplished varies widely, from ancient sun-drying methods to modern mechanical processes. Each method requires different amounts of water, time, and labor, which influences both the cost of production and the environmental impact of coffee farming. The global coffee industry processes approximately 10 million tons of coffee cherries annually, with processing methods varying by region and producer size

Rank: 2
Distance: 0.3872
Similarity: 0.6128
Confidence: 61.28%

Chunk: Different regions around the world have developed unique processing techniques based on their climate, available resources, and traditional practices. The choice of processing method can enhance certain flavor characteristics while suppressing others, making it one of the most critical decisions in coffee production. Processing begins immedia